# 📚 RAG = Retrieval-Augmented Generation

Qué pasa cuando le preguntas a un LLM algo que ocurrió después de su entrenamiento? ¿O algo que nunca estuvo en sus datos?

Hoy vamos a resolver ese problema. Construiremos un sistema que le da **memoria externa** a Gemma 4 — sin re-entrenarlo.

## 0. Configuración e imports

In [ ]:
import numpy as np
from pathlib import Path

import textwrap
import ollama

import plotly.io as pio
from IPython.display import display, Markdown
pio.renderers.default = "notebook"

from helpers.documents  import structure_pdf_with_llm, chunk_by_sections, print_chunks_preview
from helpers.embeddings import get_client, embed_text, embed_batch, save_embeddings, load_embeddings
from helpers.retrieval  import cosine_similarity, retrieve_top_k, build_rag_prompt, format_retrieval_results
from helpers.retrieval  import rag_query, compare_with_without_rag
from helpers.viz        import plot_embeddings_3d, plot_chunk_stats

MODEL = 'gemma4:e4b'

---
## 1. Del PDF al Corpus Estructurado

Antes de poder hacer RAG, necesitamos un documento limpio y bien organizado.

```
SAM33.pdf  →  [pypdf]  →  texto crudo  →  [Gemma 4]  →  texto estructurado  →  SAM33_es.txt
```

**¿Por qué no usar el PDF directamente?** `pypdf` extrae texto sin jerarquía — títulos, tablas y párrafos quedan mezclados. Usamos a Gemma 4 como **pre-procesador** para reorganizar el texto en secciones con headers markdown.

In [ ]:
PDF_PATH        = Path("local/assets/sample_documents/SAM33.pdf")
STRUCTURED_PATH = Path("local/assets/sample_documents/SAM33_es.txt")

texto_estructurado = structure_pdf_with_llm(
    pdf_path=PDF_PATH,
    output_path=STRUCTURED_PATH,
    model=MODEL,
)

In [ ]:
# Vista rápida del resultado
print("TEXTO ESTRUCTURADO (primeros 600 chars):")
print("─" * 65)
print(texto_estructurado[:600])
print("─" * 65)
print(f"\nTotal: {len(texto_estructurado):,} caracteres")

---
## 2. Chunking

Dividimos el texto en **chunks** — las unidades que vamos a indexar y recuperar.

| Opción | Problema |
|--------|----------|
| Documento completo | No cabe en el contexto, genera ruido |
| Caracteres individuales | Sin significado semántico |
| **Chunks por sección** ✅ | Cada chunk es semánticamente coherente |

In [ ]:
chunks = chunk_by_sections(texto_estructurado)

print("Chunks generados a partir de las secciones (## Subtítulos):")
print()
print_chunks_preview(chunks, n_chars=180)

In [ ]:
fig = plot_chunk_stats(chunks)
fig.show()

---
## 3. Embeddings — Texto → Vectores

Convertimos cada chunk en un **vector de 3,072 dimensiones** usando `gemini-embedding-001`. Esto nos permite medir la **similitud semántica** entre cualquier par de textos.

In [ ]:
client = get_client()

EMBEDDINGS_PATH = Path("local/embeddings/sam33_chunks.npy")

if not EMBEDDINGS_PATH.exists():
    print(f"Generando embeddings para {len(chunks)} chunks...")
    corpus_embeddings = embed_batch(
        [c["content"] for c in chunks],
        client=client,
    )
    save_embeddings(corpus_embeddings, EMBEDDINGS_PATH)
else:
    corpus_embeddings = load_embeddings(EMBEDDINGS_PATH)
    print(f"✅ Embeddings cargados: {EMBEDDINGS_PATH}  {corpus_embeddings.shape}")

In [ ]:
labels = [c["section"][:30] for c in chunks]

fig = plot_embeddings_3d(
    corpus_embeddings,
    labels=labels,
    title="Embeddings de los Chunks — SAM33",
)
fig.show()

---
## 4. Retrieval — Búsqueda Vectorial

Dado un query, encontramos los chunks más relevantes con **similitud coseno**:

$$\text{sim}(\vec{q}, \vec{c}) = \frac{\vec{q} \cdot \vec{c}}{|\vec{q}| \, |\vec{c}|}$$

In [ ]:
QUERIES_DEMO = [
    "¿Cuántas clases de sonidos cardíacos clasifica el modelo?",
    "¿Qué arquitectura de red neuronal se utiliza?",
    "¿Qué resultados obtienen con la adaptación de dominio?",
]

for q in QUERIES_DEMO:
    q_embed = embed_text(q, client=client)
    resultados = retrieve_top_k(q_embed, corpus_embeddings, chunks, k=2)
    format_retrieval_results(resultados, q)
    print()

In [ ]:
import plotly.graph_objects as go

QUERY_VIZ = "¿Qué arquitectura de red neuronal se utiliza?"

query_embed_viz = embed_text(QUERY_VIZ, client=client)
todas_sims      = cosine_similarity(query_embed_viz, corpus_embeddings)

fig = go.Figure(go.Bar(
    y=[c["section"][:30] for c in chunks],
    x=todas_sims,
    orientation="h",
    marker_color=["#e15759" if s == max(todas_sims) else "#4e79a7" for s in todas_sims],
    text=[f"{s:.3f}" for s in todas_sims],
    textposition="outside",
))

fig.update_layout(
    title=f"Similitud coseno por chunk — Query: '{QUERY_VIZ[:50]}...'",
    template="plotly_dark",
    xaxis=dict(title="Similitud coseno", range=[0, 1]),
    yaxis=dict(autorange="reversed"),
    height=max(350, 35 * len(chunks)),
    margin=dict(l=200, r=60, t=70, b=40),
    plot_bgcolor="#1e1e2e",
    paper_bgcolor="#1e1e2e",
)
fig.show()

---
## 5. Generación Aumentada — El Núcleo del RAG

Aquí conectamos todo:

```
Query  →  Embedding  →  Top-K chunks  →  Prompt + Contexto  →  LLM  →  Respuesta
```

In [ ]:
SYSTEM_SAM33 = (
    "Eres un asistente experto en deep learning y auscultación cardíaca. "
    "Responde ÚNICAMENTE basándote en el contexto proporcionado (un paper científico). "
    "Si la información no está en el contexto, di 'No encontré esa información en el paper.' "
    "Responde siempre en español, de forma clara y concisa."
)

resultado = rag_query(
    "¿Cuántas clases de sonidos cardíacos clasifica el modelo?",
    corpus_embeddings, chunks,
    model=MODEL,
    system_instruction=SYSTEM_SAM33,
    client=client,
)

display(Markdown(f"### Respuesta RAG\n\n{resultado['answer']}"))

### El poder del RAG: SIN vs CON contexto

El mismo LLM, la misma pregunta. La única diferencia: el contexto recuperado.

In [ ]:
cmp = compare_with_without_rag(
    "¿Qué pasa con la precisión del modelo cuando se aplica a sujetos humanos reales?",
    corpus_embeddings, chunks,
    model=MODEL,
    system_instruction=SYSTEM_SAM33,
    client=client,
)

display(Markdown(
    f"### ❌ SIN RAG\n\n{cmp['sin_rag']}\n\n---\n\n"
    f"### ✅ CON RAG\n\n{cmp['con_rag']['answer']}"
))

---
## 6. Demo: Preguntas al sistema RAG

In [ ]:
preguntas_demo = [
    "¿Qué modelo de red neuronal se usa y cuántos parámetros tiene?",
    "¿Cómo se simula el ruido hospitalario en los datos de entrenamiento?",
    "¿Cuál es la precisión del modelo después del fine-tuning con datos humanos?",
]

for pregunta in preguntas_demo:
    print("═" * 65)
    r = rag_query(
        pregunta, corpus_embeddings, chunks,
        model=MODEL,
        system_instruction=SYSTEM_SAM33,
        client=client,
        verbose=False,
    )
    display(Markdown(f"**🔍 {pregunta}**\n\n{r['answer']}\n\n---"))

---
## ¿Qué construimos hoy?

Un sistema RAG funcional desde cero, sin librerías mágicas como LangChain o LlamaIndex.

| Componente | Herramienta | Abstracción |
|-----------|-------------|-------------|
| Extracción de texto | pypdf + Gemma 4 | `structure_pdf_with_llm()` |
| Chunking | Regex por secciones | `chunk_by_sections()` |
| Embeddings | gemini-embedding-001 | `embed_batch()` |
| Retrieval | Similitud coseno (numpy) | `retrieve_top_k()` |
| Generación | Gemma 4 (Ollama) | `rag_query()` |
| Comparativa | SIN vs CON RAG | `compare_with_without_rag()` |